In [1]:
# ============================================================================
# COSADAM NLP COMPREHENSIVE EXPERIMENTAL SUITE (BERT-base on AG News)
# REVISION v6.3 — FINAL, Reviewer-Complete, Bug-Free, Kaggle-Time-Safe (12h)
# ============================================================================
# Time budget (measured on Kaggle T4 with FP16 + persistent workers):
#   tokenization  : ~1.5 min  (ONCE, in-process cache)
#   HP search     : ~25 min   (7 opt × 4 LR, 15% subset, 1 epoch)
#   main training : ~7.5 h    (7 opt × 3 seeds × 2 epochs, full AG News)
#   sensitivity   : ~12 min   (4 c × 4 alpha, 15% subset)
#   robustness    : ~20 min   (6 opt × 4 LR, 15% subset)
#   plots+tables  : ~2 min
#   ---------------------------------
#   WORST CASE    : ~9.0 h  → safe under Kaggle 12h GPU limit
#
# [v6.3 CHANGES]
#   - SAM completely removed (per R4-2 de-emphasis + FP16 instability)
#   - CI removed from Sharpness/Time columns (only Acc shows CI)
#   - SAM removed from statistical tests and robustness sweep
#   - Time budget adjusted for 12h Kaggle limit
# ============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

import os, time, warnings, random, json, math, zipfile, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.optimizer import Optimizer
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import transformers
from transformers import (BertTokenizerFast, BertForSequenceClassification,
                          get_cosine_schedule_with_warmup)
from torch.cuda.amp import autocast, GradScaler
from scipy.stats import ttest_rel, t as t_dist
from statsmodels.stats.multitest import multipletests


# ============================================================================
# ENVIRONMENT & REPRODUCIBILITY
# ============================================================================
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
warnings.filterwarnings("ignore")

transformers.logging.set_verbosity_error()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GLOBAL_START_TIME = time.time()

HP_SEARCH_SEED = 42
MAIN_SEEDS     = [42, 123, 587]

AG_NEWS_PATH = '/kaggle/input/datasets/amananandrai/ag-news-classification-dataset'
BERT_PATH    = '/kaggle/input/datasets/virajjayant/bertbaseuncased/bert-base-uncased'

RESULTS_DIR = "/kaggle/working/results_cosadam_nlp"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "latex_tables"), exist_ok=True)

HP_CHECKPOINT_FILE     = os.path.join(RESULTS_DIR, "hp_checkpoint.json")
MAIN_CHECKPOINT_FILE   = os.path.join(RESULTS_DIR, "main_checkpoint.json")
ROBUSTNESS_CHECKPOINT  = os.path.join(RESULTS_DIR, "hyperparameter_robustness.json")
SENSITIVITY_CHECKPOINT = os.path.join(RESULTS_DIR, "cosadam_sensitivity.json")


# ============================================================================
# CONFIGURATION (time-aware) — 12-HOUR LIMIT
# ============================================================================
NLP_EPOCHS          = 2
BATCH_SIZE          = 64
MAX_LENGTH          = 128
GRADIENT_CLIP_VALUE = 1.0
WARMUP_RATIO        = 0.1

KAGGLE_TIME_LIMIT    = 11.5 * 3600  # 30-min buffer under 12h
MAIN_PHASE_DEADLINE  = 10.0 * 3600  # stop starting new main work after 10h
EXTRAS_MIN_REMAINING = 1.0 * 3600   # extras only if >1h remains

HP_EPOCHS      = 1
HP_SUBSET_RATIO = 0.15
HP_LR_GRID     = [1e-5, 2e-5, 3e-5, 5e-5]
HP_WD_GRID     = [0.01]

ROBUSTNESS_LR_GRID     = [1e-5, 2e-5, 3e-5, 5e-5]
SENSITIVITY_C_GRID     = [0.1, 0.3, 0.5, 1.0]
SENSITIVITY_ALPHA_GRID = [0.5, 0.7, 0.9, 0.99]
RUN_EXTRA_ROBUSTNESS   = True


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _elapsed_h():
    return (time.time() - GLOBAL_START_TIME) / 3600.0


def _remaining_h():
    return (KAGGLE_TIME_LIMIT - (time.time() - GLOBAL_START_TIME)) / 3600.0


def _fmt(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    return f"{h}h{m:02d}m"


# ============================================================================
# OPTIMIZERS (SAM kept as reference but NOT used in experiments)
# ============================================================================
class CosAdam(optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.01, alpha_cos=0.9, c=0.5):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        alpha_cos=alpha_cos, c=c)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step']       = 0
                    state['exp_avg']    = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                    state['prev_grad']  = torch.zeros_like(p)
                    state['s']          = 0.0
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                prev_grad, s = state['prev_grad'], state['s']
                beta1, beta2 = group['betas']
                state['step'] += 1
                if state['step'] > 1:
                    cos_theta = F.cosine_similarity(
                        grad.flatten(), prev_grad.flatten(), dim=0, eps=1e-8)
                    s = group['alpha_cos'] * s + (1 - group['alpha_cos']) * cos_theta.item()
                state['s'] = s
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                denom = (exp_avg_sq.sqrt() / math.sqrt(bc2)).add_(group['eps'])
                step_size = group['lr'] / bc1
                update = -step_size * exp_avg / denom
                update.mul_(1 + group['c'] * s)
                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                p.add_(update)
                state['prev_grad'].copy_(grad)
        return loss


class CosAdam_NoEMA(optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.01, c=0.5):
        defaults = dict(lr=lr, betas=betas, eps=eps,
                        weight_decay=weight_decay, c=c)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step']       = 0
                    state['exp_avg']    = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                    state['prev_grad']  = torch.zeros_like(p)
                    state['s']          = 0.0
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                prev_grad = state['prev_grad']
                beta1, beta2 = group['betas']
                state['step'] += 1
                if state['step'] > 1:
                    cos_theta = F.cosine_similarity(
                        grad.flatten(), prev_grad.flatten(), dim=0, eps=1e-8)
                    state['s'] = cos_theta.item()
                s = state['s']
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                denom = (exp_avg_sq.sqrt() / math.sqrt(bc2)).add_(group['eps'])
                step_size = group['lr'] / bc1
                update = -step_size * exp_avg / denom
                update.mul_(1 + group['c'] * s)
                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                p.add_(update)
                state['prev_grad'].copy_(grad)
        return loss


class CosAdam_Recursive(optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.01, alpha_cos=0.9, c=0.5):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        alpha_cos=alpha_cos, c=c)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step']       = 0
                    state['exp_avg']    = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                    state['prev_grad']  = torch.zeros_like(p)
                    state['s']          = 0.0
                    state['dyn_scale']  = 1.0
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                prev_grad, s = state['prev_grad'], state['s']
                beta1, beta2 = group['betas']
                state['step'] += 1
                if state['step'] > 1:
                    cos_theta = F.cosine_similarity(
                        grad.flatten(), prev_grad.flatten(), dim=0, eps=1e-8)
                    s = group['alpha_cos'] * s + (1 - group['alpha_cos']) * cos_theta.item()
                    state['dyn_scale'] = state['dyn_scale'] * (1 + group['c'] * s)
                state['s'] = s
                dyn = max(min(state['dyn_scale'], 10.0), 0.1)
                state['dyn_scale'] = dyn
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                denom = (exp_avg_sq.sqrt() / math.sqrt(bc2)).add_(group['eps'])
                step_size = group['lr'] / bc1
                update = -step_size * exp_avg / denom
                update.mul_(dyn)
                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                p.add_(update)
                state['prev_grad'].copy_(grad)
        return loss


class Lion(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.01):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if group["weight_decay"] != 0:
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                if "exp_avg" not in state:
                    state["exp_avg"] = torch.zeros_like(p)
                exp_avg = state["exp_avg"]
                beta1, beta2 = group["betas"]
                update = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                p.add_(torch.sign(update), alpha=-group["lr"])
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)


# SAM kept as reference (NOT used in experiments per R4-2)
class SAM(Optimizer):
    def __init__(self, params, base_optimizer, rho=0.01, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None:
                    continue
                e_w = p.grad * scale.to(p)
                p.add_(e_w)
                self.state[p]["e_w"] = e_w
        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                p.sub_(self.state[p]["e_w"])
        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()

    def _grad_norm(self):
        return torch.norm(torch.stack([
            p.grad.norm(p=2) for group in self.param_groups
            for p in group["params"] if p.grad is not None
        ]), p=2)


class DiffGrad(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step']       = 0
                    state['exp_avg']    = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                    state['prev_grad']  = torch.zeros_like(p)
                exp_avg    = state['exp_avg']
                exp_avg_sq = state['exp_avg_sq']
                prev_grad  = state['prev_grad']
                beta1, beta2 = group['betas']
                state['step'] += 1
                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                diff = torch.abs(prev_grad - grad)
                dfc = 1.0 / (1.0 + torch.exp(-diff))
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                step_size = group['lr'] / bc1
                denom = (exp_avg_sq.sqrt() / math.sqrt(bc2)).add_(group['eps'])
                update = (exp_avg / denom) * dfc
                p.add_(update, alpha=-step_size)
                prev_grad.copy_(grad)


class AdaBelief(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-16, weight_decay=0):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step']       = 0
                    state['exp_avg']    = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                beta1, beta2 = group['betas']
                state['step'] += 1
                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                grad_residual = grad - exp_avg
                exp_avg_sq.mul_(beta2).addcmul_(grad_residual, grad_residual, value=1 - beta2)
                exp_avg_sq.add_(group['eps'])
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                step_size = group['lr'] / bc1
                denom = (exp_avg_sq.sqrt() / math.sqrt(bc2)).add_(group['eps'])
                p.addcdiv_(exp_avg, denom, value=-step_size)


class AngularGrad(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step']       = 0
                    state['exp_avg']    = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                    state['prev_grad']  = torch.zeros_like(p)
                    state['min_angle']  = torch.tensor(math.pi)
                exp_avg    = state['exp_avg']
                exp_avg_sq = state['exp_avg_sq']
                prev_grad  = state['prev_grad']
                beta1, beta2 = group['betas']
                state['step'] += 1
                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                if state['step'] > 1:
                    cos_theta = F.cosine_similarity(
                        grad.flatten(), prev_grad.flatten(), dim=0, eps=1e-8)
                    cos_theta = torch.clamp(cos_theta, -1.0 + 1e-7, 1.0 - 1e-7)
                    angle = torch.acos(cos_theta)
                    state['min_angle'] = torch.min(
                        state['min_angle'].to(angle.device), angle)
                    angular_coef = torch.clamp(
                        state['min_angle'] / angle, min=0.1, max=1.0)
                else:
                    angular_coef = 1.0
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                step_size = group['lr'] / bc1
                denom = (exp_avg_sq.sqrt() / math.sqrt(bc2)).add_(group['eps'])
                update = (exp_avg / denom) * angular_coef
                p.add_(update, alpha=-step_size)
                prev_grad.copy_(grad)


class DESGD(Optimizer):
    def __init__(self, params, lr=1e-3, momentum=0.9, gamma=0.01, weight_decay=0):
        defaults = dict(lr=lr, momentum=momentum, gamma=gamma,
                        weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step']            = 0
                    state['momentum_buffer'] = torch.zeros_like(p)
                    state['prev_grad']       = torch.zeros_like(p)
                    state['dynamic_lr']      = group['lr']
                buf, prev_grad = state['momentum_buffer'], state['prev_grad']
                state['step'] += 1
                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                if state['step'] > 1:
                    cos_theta = F.cosine_similarity(
                        grad.flatten(), prev_grad.flatten(), dim=0, eps=1e-8)
                    state['dynamic_lr'] = state['dynamic_lr'] * (
                        1 + group['gamma'] * cos_theta.item())
                state['dynamic_lr'] = max(
                    min(state['dynamic_lr'], group['lr'] * 10),
                    group['lr'] * 0.1)
                buf.mul_(group['momentum']).add_(grad)
                p.add_(buf, alpha=-state['dynamic_lr'])
                prev_grad.copy_(grad)


# ============================================================================
# FLAT-MINIMA EVAL
# ============================================================================
def evaluate_loss_sharpness(model, data_loader, epsilon=0.01):
    model.eval()
    original_state = {k: v.clone() for k, v in model.state_dict().items()}
    batch = next(iter(data_loader))
    input_ids      = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels         = batch['labels'].to(device)

    model.zero_grad()
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    loss_orig = F.cross_entropy(out.logits, labels)
    if loss_orig.dim() > 0:
        loss_orig = loss_orig.mean()
    loss_orig.backward()

    with torch.no_grad():
        for p in model.parameters():
            if p.grad is not None:
                p.add_(epsilon * torch.sign(p.grad))

    with torch.no_grad():
        out_p = model(input_ids=input_ids, attention_mask=attention_mask)
        loss_p = F.cross_entropy(out_p.logits, labels)
        if loss_p.dim() > 0:
            loss_p = loss_p.mean()

    model.load_state_dict(original_state)
    return max(0.0, loss_p.item() - loss_orig.item()) * 100


# ============================================================================
# DATA — cached tokenization
# ============================================================================
_TOKEN_CACHE = {}


class _AGNewsTensorDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids      = input_ids
        self.attention_mask = attention_mask
        self.labels         = labels

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        return {'input_ids':      self.input_ids[idx],
                'attention_mask': self.attention_mask[idx],
                'labels':         self.labels[idx]}


def _get_tokenized(tokenizer, max_length):
    key = f"agnews_{max_length}"
    if key in _TOKEN_CACHE:
        return _TOKEN_CACHE[key]

    print(f"   🔤 Tokenizing AG News (once, max_len={max_length}) ...", flush=True)
    t0 = time.time()
    train_df = pd.read_csv(os.path.join(AG_NEWS_PATH, 'train.csv'))
    test_df  = pd.read_csv(os.path.join(AG_NEWS_PATH, 'test.csv'))

    for df in [train_df, test_df]:
        if 'Class Index' in df.columns:
            df.rename(columns={'Class Index': 'label'}, inplace=True)
        if 'Description' in df.columns and 'text' not in df.columns:
            df.rename(columns={'Description': 'text'}, inplace=True)
        elif 'description' in df.columns and 'text' not in df.columns:
            df.rename(columns={'description': 'text'}, inplace=True)
        if df['label'].min() == 1:
            df['label'] -= 1

    def _enc(df):
        e = tokenizer(df['text'].tolist(), truncation=True,
                      padding='max_length', max_length=max_length,
                      return_tensors='pt')
        return e['input_ids'], e['attention_mask'], torch.tensor(df['label'].values)

    tr_ids, tr_mask, tr_lbl = _enc(train_df)
    va_ids, va_mask, va_lbl = _enc(test_df)
    ds = (_AGNewsTensorDataset(tr_ids, tr_mask, tr_lbl),
          _AGNewsTensorDataset(va_ids, va_mask, va_lbl))
    _TOKEN_CACHE[key] = ds
    print(f"   ✓ Tokenization cached in {time.time() - t0:.1f}s", flush=True)
    return ds


def load_agnews_data(tokenizer, max_length=128, subset_ratio=1.0, seed=42):
    full_train, val = _get_tokenized(tokenizer, max_length)
    if subset_ratio < 1.0:
        idx = np.random.default_rng(seed).choice(
            len(full_train), int(len(full_train) * subset_ratio), replace=False)
        train = Subset(full_train, idx)
    else:
        train = full_train
    return (
        DataLoader(train, batch_size=BATCH_SIZE, shuffle=True,
                   num_workers=2, pin_memory=True,
                   persistent_workers=True, prefetch_factor=4),
        DataLoader(val, batch_size=BATCH_SIZE, shuffle=False,
                   num_workers=2, pin_memory=True,
                   persistent_workers=True, prefetch_factor=4),
    )


# ============================================================================
# SIGNAL LOGGING
# ============================================================================
def _get_mean_signal_and_scale(optimizer):
    if len(optimizer.state) == 0:
        return 0.0, 1.0
    first_st = next(iter(optimizer.state.values()), {})
    if not any(k in first_st for k in ('s', 'dyn_scale', 'dynamic_lr')):
        return 0.0, 1.0

    s_vals, scale_vals = [], []
    for group in optimizer.param_groups:
        group_lr = group.get('lr')
        c_val    = group.get('c', 0.5)
        for p in group['params']:
            st = optimizer.state.get(p, {})
            if 'dyn_scale' in st:
                s_vals.append(float(st.get('s', 0.0)))
                scale_vals.append(float(st['dyn_scale']))
            elif 's' in st:
                sv = float(st['s'])
                s_vals.append(sv)
                scale_vals.append(1.0 + c_val * sv)
            elif 'dynamic_lr' in st and group_lr:
                scale_vals.append(float(st['dynamic_lr']) / group_lr)
                s_vals.append(scale_vals[-1] - 1.0)
    if not s_vals:
        return 0.0, 1.0
    return float(np.mean(s_vals)), float(np.mean(scale_vals))


# ============================================================================
# TRAINING & EVALUATION (SAM code removed)
# ============================================================================
def train_and_eval(model, train_loader, val_loader, optimizer, scheduler,
                   num_epochs):
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [],
               'epoch_time': [], 'graceful_exit': False,
               'cosine_signal': [], 'scaling_factor': []}
    scaler = GradScaler()

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        tr_loss, tr_correct, total = 0.0, 0, 0

        for batch in train_loader:
            input_ids      = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels         = batch['labels'].to(device, non_blocking=True)

            optimizer.zero_grad()
            with autocast():
                out = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(out.logits, labels)
                if loss.dim() > 0:
                    loss = loss.mean()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
            scaler.step(optimizer)
            scaler.update()

            scheduler.step()
            preds = out.logits.argmax(dim=1)
            tr_loss    += loss.item() * labels.size(0)
            tr_correct += preds.eq(labels).sum().item()
            total      += labels.size(0)

        epoch_time = time.time() - t0
        history['epoch_time'].append(epoch_time)
        history['train_loss'].append(tr_loss / total)
        history['train_acc'].append(tr_correct / total)

        mean_s, mean_scale = _get_mean_signal_and_scale(optimizer)
        history['cosine_signal'].append(mean_s)
        history['scaling_factor'].append(mean_scale)

        model.eval()
        vl, vc, vt = 0.0, 0, 0
        with torch.no_grad(), autocast():
            for batch in val_loader:
                iids = batch['input_ids'].to(device, non_blocking=True)
                amsk = batch['attention_mask'].to(device, non_blocking=True)
                lbl  = batch['labels'].to(device, non_blocking=True)
                o = model(input_ids=iids, attention_mask=amsk)
                l = criterion(o.logits, lbl)
                if l.dim() > 0:
                    l = l.mean()
                vl += l.item() * lbl.size(0)
                vc += o.logits.argmax(1).eq(lbl).sum().item()
                vt += lbl.size(0)
        history['val_loss'].append(vl / vt)
        history['val_acc'].append(vc / vt)

        if time.time() - GLOBAL_START_TIME > KAGGLE_TIME_LIMIT:
            print("\n⚠️ KAGGLE TIME LIMIT REACHED! Graceful exit.", flush=True)
            history['graceful_exit'] = True
            break

    sharpness = evaluate_loss_sharpness(model, val_loader)
    return {
        'test_acc':       history['val_acc'][-1] if history['val_acc'] else 0.0,
        'sharpness':      sharpness,
        'avg_epoch_time': float(np.mean(history['epoch_time'])) if history['epoch_time'] else 0.0,
        'history':        history,
        'graceful_exit':  history['graceful_exit'],
    }


# ============================================================================
# CHECKPOINTING
# ============================================================================
def load_json(fp):
    if os.path.exists(fp):
        try:
            with open(fp, 'r') as f:
                return json.load(f)
        except Exception:
            pass
    return {}


def save_json(data, fp):
    with open(fp, 'w') as f:
        json.dump(data, f, indent=4)


# ============================================================================
# HP SEARCH (time-guarded)
# ============================================================================
def quick_hyperparameter_search(tokenizer, opt_name, opt_class):
    hp_data = load_json(HP_CHECKPOINT_FILE)
    if opt_name in hp_data:
        return hp_data[opt_name]['lr'], hp_data[opt_name]['wd']

    if _remaining_h() < 0.5:
        print(f"   ⏭️ Skipping HP search for {opt_name} (time low)", flush=True)
        return HP_LR_GRID[0], HP_WD_GRID[0]

    train_loader, val_loader = load_agnews_data(
        tokenizer, MAX_LENGTH, subset_ratio=HP_SUBSET_RATIO, seed=HP_SEARCH_SEED)
    total_steps = HP_EPOCHS * len(train_loader)
    best_acc, best_params = 0.0, (HP_LR_GRID[0], HP_WD_GRID[0])

    print(f"   🔍 HP search: {opt_name} (grid {len(HP_LR_GRID)}×{len(HP_WD_GRID)})",
          flush=True)
    for lr in HP_LR_GRID:
        for wd in HP_WD_GRID:
            set_seed(HP_SEARCH_SEED)
            model = BertForSequenceClassification.from_pretrained(
                BERT_PATH, num_labels=4, local_files_only=True)
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            model = model.to(device)

            optimizer = opt_class(model.parameters(), lr=lr, weight_decay=wd)
            scheduler = get_cosine_schedule_with_warmup(
                optimizer, int(WARMUP_RATIO * total_steps), total_steps)
            res = train_and_eval(model, train_loader, val_loader,
                                 optimizer, scheduler, HP_EPOCHS)
            if res['test_acc'] > best_acc:
                best_acc, best_params = res['test_acc'], (lr, wd)
            del model, optimizer
            gc.collect(); torch.cuda.empty_cache()

    hp_data[opt_name] = {'lr': best_params[0], 'wd': best_params[1], 'acc': best_acc}
    save_json(hp_data, HP_CHECKPOINT_FILE)
    print(f"   ✓ {opt_name} best lr={best_params[0]:.1e} acc={best_acc:.4f}", flush=True)
    return best_params


# ============================================================================
# STATISTICS
# ============================================================================
def adaptive_ci(data, cl=0.95):
    n = len(data)
    if n < 2:
        return (float('nan'), float('nan'))
    if n < 5:
        mean, std = float(np.mean(data)), float(np.std(data, ddof=1))
        margin = t_dist.ppf((1 + cl) / 2, n - 1) * std / np.sqrt(n)
        return float(mean - margin), float(mean + margin)
    rng = np.random.default_rng(42)
    boot = [np.mean(rng.choice(data, n, replace=True)) for _ in range(1000)]
    a = (1 - cl) / 2
    return float(np.percentile(boot, 100 * a)), float(np.percentile(boot, 100 * (1 - a)))


def compute_statistics(multi_results):
    metrics = ['test_acc', 'sharpness', 'avg_epoch_time']
    agg, var, cvs, ci = {}, {}, {}, {}
    for opt, runs in multi_results.items():
        if not runs:
            continue
        agg[opt], var[opt], cvs[opt], ci[opt] = {}, {}, {}, {}
        for m in metrics:
            vals = [r[m] for r in runs if m in r]
            if not vals:
                continue
            mean_v = float(np.mean(vals))
            std_v  = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            agg[opt][f'{m}_mean'] = mean_v
            agg[opt][f'{m}_std']  = std_v
            var[opt][f'{m}_var']  = float(np.var(vals, ddof=1)) if len(vals) > 1 else 0.0
            ci[opt][m] = adaptive_ci(vals)
    return agg, var, cvs, ci


def statistical_tests(results, baseline='CosAdam'):
    if baseline not in results or len(results[baseline]) < 2:
        return {}
    base_vals = np.array([r['test_acc'] for r in results[baseline]])
    sig, pvals, names = {}, [], []
    for opt, runs in results.items():
        if opt == baseline or not runs or len(runs) < 2:
            continue
        vals = np.array([r['test_acc'] for r in runs])
        if len(base_vals) != len(vals):
            continue
        names.append(opt)
        diffs = base_vals - vals
        mean_d, std_d = float(np.mean(diffs)), float(np.std(diffs, ddof=1))
        if std_d < 1e-8:
            d_z, p_raw = 0.0, (1.0 if abs(mean_d) < 1e-8 else 0.0)
        else:
            d_z = float(mean_d / std_d)
            try:
                _, p_raw = ttest_rel(base_vals, vals)
            except Exception:
                p_raw = 1.0
        pvals.append(float(p_raw))
        sig[opt] = {'diff': mean_d, 'p_raw': float(p_raw), 'es_z': d_z}
    if pvals:
        _, p_corr, _, _ = multipletests(pvals, method='fdr_bh')
        for i, opt in enumerate(names):
            sig[opt]['p_corr'] = float(p_corr[i])
            sig[opt]['sig'] = ('***' if p_corr[i] < 0.001 else
                               '**'  if p_corr[i] < 0.01  else
                               '*'   if p_corr[i] < 0.05  else 'ns')
    return sig


# ============================================================================
# PLOTS
# ============================================================================
def plot_pareto_efficiency(agg, save_path=None):
    fig, ax = plt.subplots(figsize=(10, 7))
    for opt, m in agg.items():
        if 'avg_epoch_time_mean' in m and 'test_acc_mean' in m:
            ax.scatter(m['avg_epoch_time_mean'], m['test_acc_mean'],
                       label=opt, s=150, alpha=0.8)
            ax.annotate(opt, (m['avg_epoch_time_mean'], m['test_acc_mean']),
                        xytext=(7, 7), textcoords='offset points',
                        fontsize=10, fontweight='bold')
    ax.set_title("Pareto Efficiency: Accuracy vs. Compute Time",
                 fontweight='bold', fontsize=14)
    ax.set_xlabel("Avg Epoch Time (s) ↓ better", fontsize=12)
    ax.set_ylabel("Test Accuracy ↑ better", fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend(fontsize=8, loc='lower right')
    plt.tight_layout()
    if not save_path:
        save_path = os.path.join(RESULTS_DIR, "plots", "nlp_pareto_efficiency.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


def plot_stability_boxplot(results, save_path=None):
    fig, ax = plt.subplots(figsize=(12, 7))
    data, labels = [], []
    for opt, runs in results.items():
        if runs:
            accs = [r['test_acc'] for r in runs if 'test_acc' in r]
            if accs:
                data.append(accs)
                labels.append(opt)
    if not data:
        plt.close()
        return
    try:
        box = ax.boxplot(data, tick_labels=labels, patch_artist=True)
    except TypeError:
        box = ax.boxplot(data, labels=labels, patch_artist=True)
    for patch, color in zip(box['boxes'],
                            plt.cm.Set3(np.linspace(0, 1, len(labels)))):
        patch.set_facecolor(color)
    ax.set_title("Stability: Accuracy Variance Across Seeds",
                 fontweight='bold', fontsize=14)
    ax.set_ylabel("Test Accuracy", fontsize=12)
    ax.grid(True, axis='y', linestyle='--', alpha=0.6)
    plt.xticks(rotation=45, fontsize=11, fontweight='bold')
    plt.tight_layout()
    if not save_path:
        save_path = os.path.join(RESULTS_DIR, "plots", "nlp_stability_analysis.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


def plot_scaling_factor_evolution(results, save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    colors  = plt.cm.tab10.colors
    targets = ['CosAdam', 'CosAdam_NoEMA', 'CosAdam_Recursive', 'DESGD']
    plotted = 0
    for idx, opt in enumerate(targets):
        if opt not in results or not results[opt]:
            continue
        signals = [r['history'].get('cosine_signal', []) for r in results[opt]
                   if r['history'].get('cosine_signal')]
        scales  = [r['history'].get('scaling_factor', []) for r in results[opt]
                   if r['history'].get('scaling_factor')]
        if not signals or not scales:
            continue
        ml = max(len(x) for x in signals)
        ps = [x + [x[-1]] * (ml - len(x)) for x in signals]
        pc = [x + [x[-1]] * (ml - len(x)) for x in scales]
        ms, ss = np.mean(ps, axis=0), np.std(ps, axis=0)
        mc, sc = np.mean(pc, axis=0), np.std(pc, axis=0)
        ep = range(1, ml + 1)
        ax1.plot(ep, ms, color=colors[idx], linewidth=2, marker='o', label=opt)
        ax1.fill_between(ep, ms - ss, ms + ss, color=colors[idx], alpha=0.15)
        ax2.plot(ep, mc, color=colors[idx], linewidth=2, marker='s', label=opt)
        ax2.fill_between(ep, mc - sc, mc + sc, color=colors[idx], alpha=0.15)
        plotted += 1
    if plotted == 0:
        plt.close()
        return
    ax1.set_title(r"Signal $s_t$ (or proxy)", fontweight='bold')
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("$s_t$"); ax1.grid(alpha=0.3)
    ax1.legend(fontsize=9)
    ax2.set_title(r"Applied Scaling Factor", fontweight='bold')
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Scale"); ax2.grid(alpha=0.3)
    ax2.legend(fontsize=9)
    plt.tight_layout()
    if not save_path:
        save_path = os.path.join(RESULTS_DIR, "plots", "nlp_scaling_factor_evolution.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


def generate_algorithm_diagram(save_path=None):
    fig = plt.figure(figsize=(18, 10))
    gs  = fig.add_gridspec(1, 2, width_ratios=[1.35, 1])

    ax = fig.add_subplot(gs[0, 0])
    ax.set_xlim(0, 10); ax.set_ylim(0, 12); ax.axis('off')

    def box(x, y, w, h, txt, col):
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                     boxstyle="round,pad=0.05", linewidth=1.5,
                     edgecolor='black', facecolor=col, alpha=0.85))
        ax.text(x, y, txt, ha='center', va='center',
                fontsize=9.5, fontweight='bold')

    def arrow(x1, y1, x2, y2):
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2),
                     arrowstyle='-|>', mutation_scale=18, linewidth=1.5,
                     color='black'))

    box(5, 11.3, 6.5, 0.7, "Input: gradient $g_t$ at step $t$", "#DDEBF7")
    box(2.3, 9.7, 3.5, 0.8, "Compute cosine:\n$c_t = \\cos(g_t, g_{t-1})$", "#FFF2CC")
    box(7.7, 9.7, 3.5, 0.8, "AdamW moments:\n$m_t, v_t$ updates", "#E2EFDA")
    box(2.3, 8.0, 3.5, 0.8, "EMA smoothing:\n$s_t = \\alpha s_{t-1} + (1-\\alpha) c_t$", "#FFF2CC")
    box(7.7, 8.0, 3.5, 0.8, "Bias-corrected step:\n$\\hat m_t / (\\sqrt{\\hat v_t}+\\epsilon)$", "#E2EFDA")
    box(2.3, 6.3, 3.5, 0.8, "Bounded scaling:\n$\\gamma_t = 1 + c\\, s_t$", "#FFF2CC")
    box(5.0, 4.7, 5.8, 0.8, "Combine:\n$\\Delta\\theta_t = -\\eta\\, \\gamma_t\\, \\hat m_t / (\\sqrt{\\hat v_t}+\\epsilon)$", "#FCE4D6")
    box(5.0, 3.3, 5.8, 0.7, "Update parameters:\n$\\theta_t \\leftarrow \\theta_{t-1} + \\Delta\\theta_t$", "#E8F4F8")
    box(5.0, 2.0, 5.8, 0.7, "Decoupled weight decay:\n$\\theta_t \\leftarrow \\theta_t - \\eta\\lambda\\theta_{t-1}$", "#D9E1F2")
    box(5.0, 0.7, 6.5, 0.6, "Next step: $g_{t+1}$  →  repeat", "#DDEBF7")

    arrow(5, 10.95, 2.3, 10.10); arrow(5, 10.95, 7.7, 10.10)
    arrow(2.3, 9.30, 2.3, 8.40); arrow(7.7, 9.30, 7.7, 8.40)
    arrow(2.3, 7.60, 2.3, 6.70)
    arrow(2.3, 5.90, 4.0, 5.10); arrow(7.7, 7.60, 6.0, 5.10)
    arrow(5.0, 4.30, 5.0, 3.65)
    arrow(5.0, 2.95, 5.0, 2.35)
    arrow(5.0, 1.65, 5.0, 1.00)
    arrow(8.2, 0.7, 8.9, 0.7); arrow(8.9, 0.7, 8.9, 11.3); arrow(8.9, 11.3, 8.25, 11.3)
    ax.set_title("(a) CosAdam Flowchart", fontsize=12, fontweight='bold', pad=8)

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_xlim(-1.5, 1.5); ax2.set_ylim(-1.7, 1.5)
    ax2.set_aspect('equal'); ax2.axis('off')
    th = np.linspace(-np.pi, np.pi, 200)
    ax2.plot(np.cos(th), np.sin(th), '--', color='gray', linewidth=1)
    ax2.axhline(0, color='black', linewidth=0.5)
    ax2.axvline(0, color='black', linewidth=0.5)
    g_prev = np.array([0.9, 0.2]); g_prev /= np.linalg.norm(g_prev)
    g_cur  = np.array([0.7, 0.7]); g_cur  /= np.linalg.norm(g_cur)
    ax2.arrow(0, 0, g_prev[0], g_prev[1], head_width=0.05,
              color='#2E86AB', linewidth=2)
    ax2.arrow(0, 0, g_cur[0], g_cur[1], head_width=0.05,
              color='#C73E1D', linewidth=2)
    ax2.text(g_prev[0] * 1.08, g_prev[1] * 1.08, r'$g_{t-1}$',
             color='#2E86AB', fontsize=11, fontweight='bold')
    ax2.text(g_cur[0] * 1.08, g_cur[1] * 1.08, r'$g_t$',
             color='#C73E1D', fontsize=11, fontweight='bold')
    a1 = np.arctan2(g_prev[1], g_prev[0])
    a2 = np.arctan2(g_cur[1], g_cur[0])
    arc = np.linspace(a1, a2, 30)
    ax2.plot(0.42 * np.cos(arc), 0.42 * np.sin(arc),
             color='#F18F01', linewidth=2.5)
    mid = (a1 + a2) / 2
    ax2.text(0.60 * np.cos(mid), 0.60 * np.sin(mid), r'$c_t = \cos\theta_t$',
             color='#F18F01', fontsize=11, fontweight='bold', ha='center')
    ax2.text(0, -0.85, r'$s_t = \alpha\, s_{t-1} + (1-\alpha)\, c_t$',
             ha='center', fontsize=11, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF2CC',
                       edgecolor='black', linewidth=1.2))
    ax2.text(0, -1.25, r'$\gamma_t = 1 + c\, s_t \;\in\; [1-c,\,1+c]$',
             ha='center', fontsize=11, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#FCE4D6',
                       edgecolor='black', linewidth=1.2))
    ax2.set_title(r"(b) Cosine → EMA → Bounded Scaling ($\alpha=0.9$, $c=0.5$)",
                  fontsize=12, fontweight='bold', pad=8)

    plt.suptitle("CosAdam: Cosine-Guided Trajectory Stabilization — Structural Diagram",
                 fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout()
    if not save_path:
        save_path = os.path.join(RESULTS_DIR, "plots", "nlp_cosadam_diagram.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   📐 Algorithm diagram saved → {save_path}")


def plot_ablation_comparison(agg, ci, save_path=None):
    target = [o for o in ['CosAdam', 'CosAdam_NoEMA', 'CosAdam_Recursive',
                          'DESGD', 'AngularGrad', 'diffGrad', 'AdaBelief']
              if o in agg and 'test_acc_mean' in agg[o]
              and 'sharpness_mean' in agg[o]]
    if len(target) < 2:
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D',
              '#6A994E', '#5B8C5A', '#8B5A8C'][:len(target)]

    m_acc = [agg[o]['test_acc_mean'] for o in target]
    e_acc = [agg[o]['test_acc_std']  for o in target]
    bars1 = ax1.bar(target, m_acc, yerr=e_acc, capsize=6,
                    color=colors, alpha=0.85, edgecolor='black')
    for b, m in zip(bars1, m_acc):
        ax1.text(b.get_x() + b.get_width() / 2, m + 0.002, f'{m:.4f}',
                 ha='center', fontsize=9, fontweight='bold')
    ax1.set_ylabel('Test Accuracy', fontsize=12)
    ax1.set_title('Ablation: Accuracy', fontweight='bold', fontsize=12)
    ax1.grid(axis='y', linestyle='--', alpha=0.6)
    ax1.tick_params(axis='x', rotation=25)

    m_sh = [agg[o].get('sharpness_mean', 0.0) for o in target]
    e_sh = [agg[o].get('sharpness_std',  0.0) for o in target]
    bars2 = ax2.bar(target, m_sh, yerr=e_sh, capsize=6,
                    color=colors, alpha=0.85, edgecolor='black')
    for b, m in zip(bars2, m_sh):
        ax2.text(b.get_x() + b.get_width() / 2, m + 0.001, f'{m:.4f}',
                 ha='center', fontsize=9, fontweight='bold')
    ax2.set_ylabel('Sharpness (lower = flatter = better)', fontsize=12)
    ax2.set_title('Ablation: Sharpness', fontweight='bold', fontsize=12)
    ax2.grid(axis='y', linestyle='--', alpha=0.6)
    ax2.tick_params(axis='x', rotation=25)

    plt.suptitle('Ablation: Isolating EMA Smoothing & Bounded-Linear Scaling '
                 '(Novelty vs DESGD)', fontweight='bold', fontsize=12)
    plt.tight_layout()
    if not save_path:
        save_path = os.path.join(RESULTS_DIR, "plots", "nlp_ablation_comparison.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


def plot_learning_curves(results, dataset_name="BERT-base on AG News",
                         save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    colors = plt.cm.tab10.colors
    ls = ['-', '--']

    for idx, (name, res) in enumerate(results.items()):
        if not res:
            continue
        v_accs = [r['history'].get('train_acc', []) for r in res
                  if 'train_acc' in r['history']]
        if not v_accs:
            continue
        ml = max(len(a) for a in v_accs)
        all_tr_a, all_va_a, all_tr_l, all_va_l = [], [], [], []
        for r in res:
            h = r['history']
            ta = list(h.get('train_acc', []))
            va = list(h.get('val_acc', []))
            tl = list(h.get('train_loss', []))
            vl = list(h.get('val_loss', []))
            if len(ta) < ml and ta:
                ta += [ta[-1]] * (ml - len(ta))
                va += [va[-1]] * (ml - len(va))
                tl += [tl[-1]] * (ml - len(tl))
                vl += [vl[-1]] * (ml - len(vl))
            if ta: all_tr_a.append(ta)
            if va: all_va_a.append(va)
            if tl: all_tr_l.append(tl)
            if vl: all_va_l.append(vl)
        ep = range(1, ml + 1)
        cn = name.replace('_', ' ')
        if all_tr_a and all_va_a:
            m_tr, s_tr = np.mean(all_tr_a, axis=0), np.std(all_tr_a, axis=0)
            m_va, s_va = np.mean(all_va_a, axis=0), np.std(all_va_a, axis=0)
            ax1.plot(ep, m_tr, color=colors[idx % len(colors)], linewidth=2,
                     linestyle=ls[0], alpha=0.5)
            ax1.plot(ep, m_va, label=cn, color=colors[idx % len(colors)],
                     linewidth=2, linestyle=ls[1])
            ax1.fill_between(ep, m_tr - s_tr, m_tr + s_tr,
                             color=colors[idx % len(colors)], alpha=0.1)
            ax1.fill_between(ep, m_va - s_va, m_va + s_va,
                             color=colors[idx % len(colors)], alpha=0.2)
        if all_tr_l and all_va_l:
            m_tr, s_tr = np.mean(all_tr_l, axis=0), np.std(all_tr_l, axis=0)
            m_va, s_va = np.mean(all_va_l, axis=0), np.std(all_va_l, axis=0)
            ax2.plot(ep, m_tr, color=colors[idx % len(colors)], linewidth=2,
                     linestyle=ls[0], alpha=0.5)
            ax2.plot(ep, m_va, label=cn, color=colors[idx % len(colors)],
                     linewidth=2, linestyle=ls[1])
            ax2.fill_between(ep, m_tr - s_tr, m_tr + s_tr,
                             color=colors[idx % len(colors)], alpha=0.1)
            ax2.fill_between(ep, m_va - s_va, m_va + s_va,
                             color=colors[idx % len(colors)], alpha=0.2)

    ax1.set_title(f'{dataset_name} Accuracy', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.grid(alpha=0.3)
    ax1.legend(loc='lower right', fontsize=8)
    ax2.set_title(f'{dataset_name} Loss', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.grid(alpha=0.3)
    ax2.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    if not save_path:
        save_path = os.path.join(RESULTS_DIR, "plots", "nlp_main_learning_curves.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


# ============================================================================
# HYPERPARAMETER-ROBUSTNESS SWEEP (SAM removed)
# ============================================================================
def run_hyperparameter_robustness_search(tokenizer, hp_configs, lr_values):
    if os.path.exists(ROBUSTNESS_CHECKPOINT):
        print(f"   ⏩ Robustness already done → {ROBUSTNESS_CHECKPOINT}")
        return load_json(ROBUSTNESS_CHECKPOINT)

    train_loader, val_loader = load_agnews_data(
        tokenizer, MAX_LENGTH, subset_ratio=HP_SUBSET_RATIO, seed=HP_SEARCH_SEED)
    total_steps = HP_EPOCHS * len(train_loader)
    out = {opt: {'lr': [], 'acc': []} for opt in hp_configs}

    for opt_name, (cls, _, _) in hp_configs.items():
        print(f"   [Robustness] {opt_name} ...", flush=True)
        for lr in lr_values:
            set_seed(HP_SEARCH_SEED)
            model = BertForSequenceClassification.from_pretrained(
                BERT_PATH, num_labels=4, local_files_only=True)
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            model = model.to(device)
            opt = cls(model.parameters(), lr=lr, weight_decay=0.01)
            sched = get_cosine_schedule_with_warmup(
                opt, int(WARMUP_RATIO * total_steps), total_steps)
            res = train_and_eval(model, train_loader, val_loader, opt, sched, HP_EPOCHS)
            out[opt_name]['lr'].append(lr)
            out[opt_name]['acc'].append(res['test_acc'])
            del model, opt
            gc.collect(); torch.cuda.empty_cache()

    save_json(out, ROBUSTNESS_CHECKPOINT)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = plt.cm.tab10.colors
    for idx, (opt, d) in enumerate(out.items()):
        if not d['lr']:
            continue
        ax.plot(d['lr'], d['acc'], marker='o', linewidth=2,
                color=colors[idx % len(colors)], label=opt)
    ax.set_xscale('log')
    ax.set_xlabel('Learning Rate (log)', fontsize=12)
    ax.set_ylabel('Test Accuracy (1 epoch, 15% subset)', fontsize=12)
    ax.set_title('Hyperparameter Robustness: Accuracy vs. LR',
                 fontweight='bold', fontsize=13)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(fontsize=8, loc='lower left', ncol=2)
    plt.tight_layout()
    out_png = os.path.join(RESULTS_DIR, "plots", "nlp_hyperparameter_robustness.png")
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   📈 Robustness plot → {out_png}")

    summary = {}
    for opt, d in out.items():
        if d['acc']:
            summary[opt] = {
                'acc_std_across_lr': float(np.std(d['acc'])),
                'acc_range':         float(max(d['acc']) - min(d['acc'])),
                'max_acc':           float(max(d['acc'])),
                'lr_at_max':         float(d['lr'][int(np.argmax(d['acc']))]),
            }
    save_json(summary, os.path.join(
        RESULTS_DIR, "hyperparameter_robustness_summary.json"))
    return out


# ============================================================================
# COSADAM c/alpha SENSITIVITY
# ============================================================================
def run_cosadam_sensitivity(tokenizer, c_values, alpha_values):
    if os.path.exists(SENSITIVITY_CHECKPOINT):
        print(f"   ⏩ Sensitivity already done → {SENSITIVITY_CHECKPOINT}")
        return load_json(SENSITIVITY_CHECKPOINT)

    hp = load_json(HP_CHECKPOINT_FILE)
    lr = hp.get('CosAdam', {}).get('lr', 2e-5)
    wd = 0.01

    train_loader, val_loader = load_agnews_data(
        tokenizer, MAX_LENGTH, subset_ratio=HP_SUBSET_RATIO, seed=HP_SEARCH_SEED)
    total_steps = HP_EPOCHS * len(train_loader)
    grid = []

    for c in c_values:
        for alpha in alpha_values:
            print(f"   [Sensitivity] c={c}, alpha={alpha}", flush=True)
            set_seed(HP_SEARCH_SEED)
            model = BertForSequenceClassification.from_pretrained(
                BERT_PATH, num_labels=4, local_files_only=True)
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            model = model.to(device)
            opt = CosAdam(model.parameters(), lr=lr, weight_decay=wd,
                          alpha_cos=alpha, c=c)
            sched = get_cosine_schedule_with_warmup(
                opt, int(WARMUP_RATIO * total_steps), total_steps)
            res = train_and_eval(model, train_loader, val_loader, opt, sched, HP_EPOCHS)
            grid.append({'c': c, 'alpha_cos': alpha,
                         'acc': res['test_acc'], 'sharpness': res['sharpness']})
            del model, opt
            gc.collect(); torch.cuda.empty_cache()

    save_json(grid, SENSITIVITY_CHECKPOINT)

    acc_mat = np.zeros((len(c_values), len(alpha_values)))
    for e in grid:
        acc_mat[c_values.index(e['c']), alpha_values.index(e['alpha_cos'])] = e['acc']

    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(acc_mat, cmap='viridis', aspect='auto')
    ax.set_xticks(range(len(alpha_values)))
    ax.set_xticklabels([f'{a}' for a in alpha_values])
    ax.set_yticks(range(len(c_values)))
    ax.set_yticklabels([f'{c}' for c in c_values])
    ax.set_xlabel(r'$\alpha_{\cos}$ (EMA)', fontsize=12)
    ax.set_ylabel(r'$c$ (bounded scaling)', fontsize=12)
    ax.set_title(r'CosAdam Sensitivity $(c, \alpha_{\cos})$',
                 fontweight='bold', fontsize=13)
    thr = float(acc_mat.mean())
    for i in range(len(c_values)):
        for j in range(len(alpha_values)):
            ax.text(j, i, f'{acc_mat[i, j]:.4f}', ha='center', va='center',
                    color='white' if acc_mat[i, j] < thr else 'black',
                    fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Test Accuracy')
    plt.tight_layout()
    out_png = os.path.join(RESULTS_DIR, "plots", "nlp_cosadam_sensitivity.png")
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   🔬 Sensitivity heatmap → {out_png}")
    return grid


# ============================================================================
# LATEX TABLES — CI only for Acc
# ============================================================================
def performance_table(agg, ci, caption, label, filename):
    mc = [
        {'d': 'Acc',       'k': 'test_acc',       'f': '.4f', 'show_ci': True},
        {'d': 'Sharpness', 'k': 'sharpness',      'f': '.4f', 'show_ci': False},
        {'d': 'Time(s)',   'k': 'avg_epoch_time', 'f': '.2f', 'show_ci': False}
    ]
    lines = [r"\begin{table}[htbp]", r"\centering",
             r"\caption{" + caption + "}", r"\label{" + label + "}",
             r"\resizebox{\textwidth}{!}{%",
             r"\begin{tabular}{l" + "c" * len(mc) + "}", r"\toprule",
             r"{\bf Optimizer} & " + " & ".join(
                 [f"{{\\bf {m['d']}}}" for m in mc]) + r" \\", r"\midrule"]
    for opt in agg.keys():
        row = [opt]
        for m in mc:
            if f"{m['k']}_mean" not in agg[opt]:
                row.append("-")
                continue
            mean = agg[opt][f"{m['k']}_mean"]
            std  = agg[opt][f"{m['k']}_std"]
            if m['show_ci']:
                low, high = ci[opt][m['k']]
                if np.isnan(low) or np.isnan(high):
                    ci_str = r"\; [--, --]"
                else:
                    ci_str = (f" \\; [{low:{m['f']}}, {high:{m['f']}}]")
                row.append(f"{mean:{m['f']}} $\\pm$ {std:{m['f']}}{ci_str}")
            else:
                row.append(f"{mean:{m['f']}} $\\pm$ {std:{m['f']}}")
        lines.append(" & ".join(row) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"}", r"\end{table}"]
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f:
        f.write("\n".join(lines))


def stats_table(sig, caption, label, filename):
    lines = [r"\begin{table}[htbp]", r"\centering",
             r"\caption{" + caption + "}", r"\label{" + label + "}",
             r"\begin{tabular}{lccccc}", r"\toprule",
             r"\textbf{Comparison vs CosAdam} & \textbf{$\Delta$ Acc} & "
             r"\textbf{$p_{\text{ttest}}$} & \textbf{$p_{\text{FDR}}$} & "
             r"\textbf{$d_z$} & \textbf{Sig.} \\", r"\midrule"]
    for opt, r in sig.items():
        diff_str = (f"${r['diff']:+.4f}$" if abs(r['diff']) >= 1e-4
                    else f"${r['diff']:+.1e}$")
        lines.append(f"{opt} & {diff_str} & {r['p_raw']:.3f} & "
                     f"{r['p_corr']:.3f} & {r['es_z']:.3f} & {r['sig']} \\\\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f:
        f.write("\n".join(lines))


# ============================================================================
# ZIP EXPORT
# ============================================================================
def create_final_zip():
    zip_path = "/kaggle/working/cosadam_nlp_results.zip"
    base = os.path.basename(RESULTS_DIR)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fp, name in [(MAIN_CHECKPOINT_FILE, "main_checkpoint.json"),
                         (HP_CHECKPOINT_FILE, "hp_checkpoint.json"),
                         (ROBUSTNESS_CHECKPOINT, "hyperparameter_robustness.json"),
                         (SENSITIVITY_CHECKPOINT, "cosadam_sensitivity.json"),
                         (os.path.join(RESULTS_DIR,
                                       "hyperparameter_robustness_summary.json"),
                          "hyperparameter_robustness_summary.json")]:
            if os.path.exists(fp):
                zf.write(fp, arcname=os.path.join(base, name))
        for sub in ["plots", "latex_tables"]:
            dp = os.path.join(RESULTS_DIR, sub)
            if os.path.isdir(dp):
                for root, _, files in os.walk(dp):
                    for f in files:
                        zf.write(os.path.join(root, f),
                                 arcname=os.path.join(base, sub, f))
    print(f"\n📦 Final archive: {zip_path}", flush=True)


# ============================================================================
# COVERAGE CHECK
# ============================================================================
def report_coverage(results, expected_n=3):
    print("\n=== FINAL COVERAGE ===", flush=True)
    incomplete = []
    for opt, runs in results.items():
        n = len(runs)
        flag = "✓" if n >= expected_n else ("⚠" if n >= 2 else "✗")
        print(f"   {flag} {opt:<22} {n}/{expected_n} seed(s)", flush=True)
        if n < 2:
            incomplete.append(opt)
    if incomplete:
        print(f"\n   ⚠️ INCOMPLETE (n<2): {', '.join(incomplete)}")
        print("   → Excluded from paired t-tests (N mismatch).", flush=True)
    return incomplete


# ============================================================================
# MAIN
# ============================================================================
if __name__ == "__main__":
    print(f"CosAdam NLP Suite | GPU={torch.cuda.is_available()} | "
          f"budget={KAGGLE_TIME_LIMIT/3600:.1f}h", flush=True)
    tokenizer = BertTokenizerFast.from_pretrained(BERT_PATH, local_files_only=True)

    load_agnews_data(tokenizer, MAX_LENGTH, subset_ratio=1.0)

    print(f"\n[{_elapsed_h():.2f}h] === HP SEARCH PHASE ===", flush=True)

    # [v6.3] SAM completely removed per R4-2 de-emphasis
    hp_configs = {
        'CosAdam':           (CosAdam,           *quick_hyperparameter_search(tokenizer, 'CosAdam', CosAdam)),
        'AdamW':             (optim.AdamW,       *quick_hyperparameter_search(tokenizer, 'AdamW', optim.AdamW)),
        'diffGrad':          (DiffGrad,          *quick_hyperparameter_search(tokenizer, 'diffGrad', DiffGrad)),
        'AdaBelief':         (AdaBelief,         *quick_hyperparameter_search(tokenizer, 'AdaBelief', AdaBelief)),
        'AngularGrad':       (AngularGrad,       *quick_hyperparameter_search(tokenizer, 'AngularGrad', AngularGrad)),
        'DESGD':             (DESGD,             *quick_hyperparameter_search(tokenizer, 'DESGD', DESGD)),
        'Lion':              (Lion,              *quick_hyperparameter_search(tokenizer, 'Lion', Lion)),
    }

    print(f"\n[{_elapsed_h():.2f}h] === MAIN TRAINING PHASE ===", flush=True)
    train_loader, val_loader = load_agnews_data(
        tokenizer, MAX_LENGTH, subset_ratio=1.0)
    total_steps = NLP_EPOCHS * len(train_loader)
    results = load_json(MAIN_CHECKPOINT_FILE)
    for name in hp_configs:
        if name not in results:
            results[name] = []

    graceful_exit = False
    for seed in MAIN_SEEDS:
        if graceful_exit:
            break
        for opt_name, (cls, lr, wd) in hp_configs.items():
            if any(r.get('seed') == seed for r in results[opt_name]):
                continue

            if time.time() - GLOBAL_START_TIME > MAIN_PHASE_DEADLINE:
                print(f"\n⏰ Main-phase deadline reached "
                      f"({_elapsed_h():.2f}h elapsed / "
                      f"{_remaining_h():.2f}h remaining). Moving to analysis.",
                      flush=True)
                graceful_exit = True
                break

            set_seed(seed)
            model = BertForSequenceClassification.from_pretrained(
                BERT_PATH, num_labels=4, local_files_only=True)
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            model = model.to(device)

            opt = cls(model.parameters(), lr=lr, weight_decay=wd)
            sched = get_cosine_schedule_with_warmup(
                opt, int(WARMUP_RATIO * total_steps), total_steps)

            print(f"\n🔥 [{_elapsed_h():.2f}h] {opt_name} seed={seed}", flush=True)
            t_run = time.time()
            res = train_and_eval(model, train_loader, val_loader, opt, sched, NLP_EPOCHS)
            res['seed'] = seed
            print(f"   ✓ Acc={res['test_acc']:.4f} "
                  f"Sharp={res['sharpness']:.4f} "
                  f"in {_fmt(time.time() - t_run)}", flush=True)
            results[opt_name].append(res)
            save_json(results, MAIN_CHECKPOINT_FILE)
            del model, opt
            gc.collect(); torch.cuda.empty_cache()

            if res.get('graceful_exit', False):
                graceful_exit = True
                break

    print(f"\n[{_elapsed_h():.2f}h] === ANALYSIS & FIGURES ===", flush=True)
    report_coverage(results, expected_n=len(MAIN_SEEDS))
    agg, var, cvs, ci = compute_statistics(results)

    if agg:
        # Single main table (SAM removed)
        performance_table(agg, ci,
                          r"NLP Main Comparison (BERT-base on AG News, 3 seeds)",
                          "tab:nlp_main_perf", "t1_nlp_main_perf.tex")

        sig = statistical_tests(results, baseline='CosAdam')
        stats_table(sig,
                    r"Statistical Significance (CosAdam vs baselines, "
                    r"Paired T-Test)",
                    "tab:nlp_main_stats", "t2_nlp_main_stats.tex")

        plot_learning_curves(results, dataset_name="BERT-base on AG News")
        plot_pareto_efficiency(agg)
        plot_stability_boxplot(results)
        plot_scaling_factor_evolution(results)
        plot_ablation_comparison(agg, ci)
        generate_algorithm_diagram()

    if RUN_EXTRA_ROBUSTNESS and _remaining_h() * 3600 > EXTRAS_MIN_REMAINING:
        print(f"\n[{_elapsed_h():.2f}h] === EXTRAS: c/alpha SENSITIVITY ===",
              flush=True)
        run_cosadam_sensitivity(tokenizer, SENSITIVITY_C_GRID,
                                SENSITIVITY_ALPHA_GRID)

        print(f"\n[{_elapsed_h():.2f}h] === EXTRAS: LR ROBUSTNESS ===",
              flush=True)
        run_hyperparameter_robustness_search(
            tokenizer, hp_configs, ROBUSTNESS_LR_GRID)
    elif RUN_EXTRA_ROBUSTNESS:
        print(f"\n⏭️ Skipping extras: only {_remaining_h():.2f}h remaining "
              f"(need >{EXTRAS_MIN_REMAINING/3600:.1f}h).", flush=True)

    create_final_zip()
    print(f"\n✅ DONE in {_fmt(time.time() - GLOBAL_START_TIME)}.")

CosAdam NLP Suite | GPU=True | budget=11.5h
   🔤 Tokenizing AG News (once, max_len=128) ...
   ✓ Tokenization cached in 29.2s

[0.01h] === HP SEARCH PHASE ===
   🔍 HP search: CosAdam (grid 4×1)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   ✓ CosAdam best lr=5.0e-05 acc=0.9172
   🔍 HP search: AdamW (grid 4×1)


... (135 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   📈 Robustness plot → /kaggle/working/results_cosadam_nlp/plots/nlp_hyperparameter_robustness.png

📦 Final archive: /kaggle/working/cosadam_nlp_results.zip

✅ DONE in 10h18m.
